In [1]:
# Basic imports
import io
import json
import uuid
from pprint import pprint
from tqdm import tqdm

# Third party core
import boto3
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Sagemaker imports
import sagemaker
from sagemaker import ModelPackage
from sagemaker.deserializers import CSVDeserializer, JSONDeserializer
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DatasetFormat,
    DefaultModelMonitor,
    ModelQualityMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r

In [3]:
%store

Stored variables and their in-db values:
alarm_names                                -> ['cls-f1-141e5d13', 'cls-input-brightness-drift-14
baseline_prefix                            -> 'cat-landmarks-project/monitoring/29f23641-f5b4-4c
bucket                                     -> 'sagemaker-us-east-1-298748835671'
cls_drift_schedule                         -> 'cls-drift-hourly-141e5d13'
cls_endpoint_name                          -> 'cls-29f23641-f5b4-4c0d-80c7-8036eeb14ff0'
cls_model_data_url                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
cls_model_package_arn                      -> 'arn:aws:sagemaker:us-east-1:298748835671:model-pa
cls_quality_schedule                       -> 'cls-quality-hourly-141e5d13'
data_capture_prefix                        -> 'cat-landmarks-project/monitoring/29f23641-f5b4-4c
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                    

In [4]:
bucket = bucket
project_prefix = project_prefix

s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)

# Deploying Registered Models
Both endpoints serve the model versions that passed their pipeline's quality gate and were registered by `5_Classification_Model` and `5-1_Keypoint_Regression_Model`. Data capture records every request and response.

In [5]:
# Endpoints
endpoint_id = uuid.uuid4()
kp_endpoint_name = f"keypoint-regression-{endpoint_id}"
cls_endpoint_name = f"cls-{endpoint_id}"

# Registered model versions
kp_model = ModelPackage(
    role=role,
    model_package_arn=kp_model_package_arn,
    sagemaker_session=sagemaker_session,
)

cls_model = ModelPackage(
    role=role,
    model_package_arn=cls_model_package_arn,
    sagemaker_session=sagemaker_session,
)

# S3 locations for captured traffic and baselines
monitoring_prefix = f"{project_prefix}/monitoring/{endpoint_id}"
data_capture_prefix = f"{monitoring_prefix}/datacapture"
baseline_prefix = f"{monitoring_prefix}/baselines"

In [6]:
# Deploying Keypoint model
kp_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=kp_endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True, 
        sampling_percentage=100, 
        destination_s3_uri=f"s3://{bucket}/{data_capture_prefix}/keypoint_regressor"
    ),
)

-----------!

In [7]:
# Deploying Classification model
cls_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=cls_endpoint_name,
    data_capture_config=DataCaptureConfig(
        enable_capture=True, 
        sampling_percentage=100, 
        destination_s3_uri=f"s3://{bucket}/{data_capture_prefix}/classifier"
    ),
)

---------!

In [8]:
# Both endpoints receive raw image bytes, the way a production client would send them
kp_predictor = Predictor(
    endpoint_name=kp_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

# The classifier answers in CSV so Model Monitor's model-quality job can read the
# prediction (column 0) and cat probability (column 1) by position
cls_predictor = Predictor(
    endpoint_name=cls_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=CSVDeserializer(),
)

%store kp_endpoint_name cls_endpoint_name

Stored 'kp_endpoint_name' (str)
Stored 'cls_endpoint_name' (str)


# Baseline Traffic
Each model's held-out test split is sent through its endpoint. Every request carries a unique inference ID, which is logged with the image and its ground truth. Model Monitor uses these IDs to join captured predictions with ground-truth labels.

Baselines are computed from the endpoint's responses rather than from offline predictions, so they reflect the serving path that will be monitored.

In [9]:
def read_s3_bytes(s3_uri: str) -> bytes:
    bucket_name, key = s3_uri.replace("s3://", "").split("/", 1)
    return s3.get_object(Bucket=bucket_name, Key=key)["Body"].read()


def invoke(predictor, s3_uri: str):
    """Send raw image bytes to an endpoint, tagged with a unique inference ID."""
    inference_id = str(uuid.uuid4())
    response = predictor.predict(read_s3_bytes(s3_uri), inference_id=inference_id)
    return inference_id, response


# Request-image statistics both endpoints return with each prediction
INPUT_STAT_NAMES = ["width", "height", "aspect_ratio", "brightness", "contrast"]


def parse_classifier_csv(response) -> dict:
    """Classifier CSV response: prediction, cat probability, then INPUT_STAT_NAMES."""
    values = response[0]
    return {
        "prediction": int(values[0]),
        "probability": float(values[1]),
        **{name: float(value) for name, value in zip(INPUT_STAT_NAMES, values[2:])},
    }


def upload_csv(df: pd.DataFrame, key: str) -> str:
    s3.put_object(Bucket=bucket, Key=key, Body=df.to_csv(index=False).encode())
    return f"s3://{bucket}/{key}"


def registered_evaluation(model_package_arn: str) -> dict:
    """Evaluation report the training pipeline attached to a registered model."""
    package = sagemaker_client.describe_model_package(ModelPackageName=model_package_arn)
    return json.loads(read_s3_bytes(package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]))


def summarize_feature_statistics(baseline_job) -> pd.DataFrame:
    """One row per feature from a data-quality baseline's statistics.json."""
    rows = []
    for feature in baseline_job.baseline_statistics().body_dict["features"]:
        stats = feature.get("numerical_statistics", {})
        rows.append({
            "feature": feature["name"],
            "type": feature["inferred_type"],
            "mean": stats.get("mean"),
            "std_dev": stats.get("std_dev"),
            "min": stats.get("min"),
            "max": stats.get("max"),
        })
    return pd.DataFrame(rows).set_index("feature")

## Classifier

In [10]:
# Test split of the training manifest used by 5_Classification_Model
training_manifest = pd.read_parquet(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{training_manifest_remote_path}")))
cls_test = training_manifest[training_manifest["split"] == "test"]

rows = []
for uri, label, source in tqdm(zip(cls_test["remote_path"], cls_test["label"], cls_test["source"]),
                               total=len(cls_test), desc="Classifier baseline traffic", unit="image"):
    inference_id, response = invoke(cls_predictor, uri)
    rows.append({
        "inference_id": inference_id,
        "remote_path": uri,
        "source": source,
        "label": int(label),
        **parse_classifier_csv(response),
    })

cls_baseline_df = pd.DataFrame(rows)
cls_baseline_df.groupby(["source", "label"]).size().unstack("label")

Classifier baseline traffic:   0%|          | 0/3638 [00:00<?, ?image/s]

Classifier baseline traffic: 100%|██████████| 3638/3638 [06:42<00:00,  9.05image/s]


label,0,1
source,,
coco,1500.0,411.0
kaggle_cats,NaN,993.0
oxford_pets,497.0,237.0


In [11]:
# Train/serve consistency: the pipeline's offline evaluation and the deployed endpoint
# score the same test split. A large gap would point to preprocessing skew.
offline = registered_evaluation(cls_model_package_arn)["binary_classification_metrics"]

endpoint_metrics = {
    "accuracy": accuracy_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "precision": precision_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "recall": recall_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "f1": f1_score(cls_baseline_df["label"], cls_baseline_df["prediction"]),
    "auc": roc_auc_score(cls_baseline_df["label"], cls_baseline_df["probability"]),
}

display(pd.DataFrame({
    "pipeline_evaluation": {name: offline[name]["value"] for name in endpoint_metrics},
    "endpoint": endpoint_metrics,
}).round(4))

print(classification_report(cls_baseline_df["label"], cls_baseline_df["prediction"],
                            target_names=["non_cat", "cat"], digits=4))

# Endpoint accuracy per source, to compare with the pipeline's per-source evaluation
pd.DataFrame({
    "pipeline_accuracy": {source: m["accuracy"] for source, m in registered_evaluation(cls_model_package_arn)["per_source_metrics"].items()},
    "endpoint_accuracy": cls_baseline_df.assign(correct=cls_baseline_df["label"] == cls_baseline_df["prediction"]).groupby("source")["correct"].mean(),
}).round(4)

,pipeline_evaluation,endpoint
accuracy,0.9508,0.9508
precision,0.9692,0.9692
recall,0.9202,0.9202
f1,0.9440,0.9440
auc,0.9889,0.9889


              precision    recall  f1-score   support

     non_cat     0.9370    0.9760    0.9561      1997
         cat     0.9692    0.9202    0.9440      1641

    accuracy                         0.9508      3638
   macro avg     0.9531    0.9481    0.9501      3638
weighted avg     0.9515    0.9508    0.9507      3638



,pipeline_accuracy,endpoint_accuracy
coco,0.9257,0.9257
kaggle_cats,0.9839,0.9839
oxford_pets,0.9714,0.9714


In [12]:
cls_baseline_prefix = f"{baseline_prefix}/classifier"

# Model-quality baseline dataset: ground truth, predicted label, cat probability
cls_quality_dataset_uri = upload_csv(
    cls_baseline_df[["label", "prediction", "probability"]],
    f"{cls_baseline_prefix}/model-quality-dataset.csv",
)

# Input/prediction drift baseline dataset: request-image statistics and endpoint outputs,
# no labels needed. Columns are alphabetical, matching src/monitoring/classifier_record_preprocessor.py
CLS_DRIFT_FEATURES = sorted(["prediction", "probability"] + INPUT_STAT_NAMES)
cls_prediction_dataset_uri = upload_csv(
    cls_baseline_df[CLS_DRIFT_FEATURES].astype({"width": "int64", "height": "int64"}),
    f"{cls_baseline_prefix}/prediction-dataset.csv",
)

# Inference log: which inference ID was sent with which image and label
upload_csv(cls_baseline_df, f"{cls_baseline_prefix}/inference-log.csv")

's3://sagemaker-us-east-1-298748835671/cat-landmarks-project/monitoring/19d31439-87ca-4ff9-87e8-3327a4316c7f/baselines/classifier/inference-log.csv'

### Model-quality baseline
Metrics and suggested constraints from predictions against ground truth. Model Monitor compares live traffic to these once ground-truth labels arrive.

In [13]:
cls_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_quality_monitor.suggest_baseline(
    job_name=f"cls-quality-baseline-{endpoint_id}",
    baseline_dataset=cls_quality_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{cls_baseline_prefix}/model-quality",
    wait=True,
    logs=False,
)

INFO:sagemaker:Creating processing-job with name cls-quality-baseline-19d31439-87ca-4ff9-87e8-3327a4316c7f


............................................................!

In [14]:
cls_quality_baseline_job = cls_quality_monitor.latest_baselining_job

cls_quality_statistics = cls_quality_baseline_job.baseline_statistics().body_dict["binary_classification_metrics"]
display(pd.DataFrame(
    {name: metric for name, metric in cls_quality_statistics.items() if isinstance(metric, dict) and "value" in metric}
).T)

pd.DataFrame(cls_quality_baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

,value,standard_deviation
recall,0.920171,0.002398
precision,0.969191,0.000457
accuracy,0.950797,0.001142
recall_best_constant_classifier,0.000000,0.000000
precision_best_constant_classifier,0.000000,0.000000
accuracy_best_constant_classifier,0.548928,0.003157
true_positive_rate,0.920171,0.002398
true_negative_rate,0.975964,0.000417
false_positive_rate,0.024036,0.000417
false_negative_rate,0.079829,0.002398


,threshold,comparison_operator
recall,0.920171,LessThanThreshold
precision,0.969191,LessThanThreshold
accuracy,0.950797,LessThanThreshold
true_positive_rate,0.920171,LessThanThreshold
true_negative_rate,0.975964,LessThanThreshold
false_positive_rate,0.024036,GreaterThanThreshold
false_negative_rate,0.079829,GreaterThanThreshold
auc,0.98894,LessThanThreshold
f0_5,0.958974,LessThanThreshold
f1,0.944045,LessThanThreshold


### Input and prediction drift baseline
Distributions of the request images (size, aspect ratio, brightness, contrast) and of the endpoint's outputs (predicted class and cat probability). Drift in these needs no labels, so it can be detected as soon as traffic arrives.

In [15]:
cls_prediction_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_prediction_monitor.suggest_baseline(
    job_name=f"cls-prediction-baseline-{endpoint_id}",
    baseline_dataset=cls_prediction_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{cls_baseline_prefix}/prediction-drift",
    wait=True,
    logs=False,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name cls-prediction-baseline-19d31439-87ca-4ff9-87e8-3327a4316c7f


............................................................!

In [16]:
cls_prediction_baseline_job = cls_prediction_monitor.latest_baselining_job

display(summarize_feature_statistics(cls_prediction_baseline_job))
pprint(cls_prediction_baseline_job.suggested_constraints().body_dict["monitoring_config"])

,type,mean,std_dev,min,max
feature,,,,,
aspect_ratio,Fractional,1.237362,0.328532,0.408000,3.699422
brightness,Fractional,0.442373,0.116218,0.043037,0.915642
contrast,Fractional,0.233887,0.053785,0.048777,0.428935
height,Integral,473.413964,142.733824,108.000000,1024.000000
prediction,Integral,0.428257,0.494826,0.000000,1.000000
probability,Fractional,0.442434,0.455292,0.000006,1.000000
width,Integral,561.401319,161.540745,144.000000,1024.000000


{'datatype_check_threshold': 1.0,
 'distribution_constraints': {'categorical_comparison_threshold': 0.1,
                              'categorical_drift_method': 'LInfinity',
                              'comparison_method': 'Robust',
                              'comparison_threshold': 0.1,
                              'perform_comparison': 'Enabled'},
 'domain_content_threshold': 1.0,
 'emit_metrics': 'Enabled',
 'evaluate_constraints': 'Enabled'}


## Keypoint Regressor

In [17]:
KP_NAMES = [
    "left_eye", "right_eye", "mouth",
    "left_ear_1", "left_ear_2", "left_ear_3",
    "right_ear_1", "right_ear_2", "right_ear_3",
]

KP_COLS_NORM = [f"{kp}_{axis}_norm" for kp in KP_NAMES for axis in ("x", "y")]
KP_COLS_PRED = [f"{kp}_{axis}_pred" for kp in KP_NAMES for axis in ("x", "y")]


def keypoint_nme(pred_norm, gt_norm, width, height) -> np.ndarray:
    """Per-keypoint pixel error divided by the ground-truth inter-ocular distance.
    Same definition as src/keypoint_regression/evaluation.py."""
    scale = np.array([width, height], dtype=float)
    pred_px = np.asarray(pred_norm, dtype=float).reshape(-1, 2) * scale
    gt_px = np.asarray(gt_norm, dtype=float).reshape(-1, 2) * scale
    inter_ocular = np.linalg.norm(gt_px[0] - gt_px[1])
    if inter_ocular == 0:
        # Coinciding eye annotations give no usable scale; excluded, as in evaluation.py
        return np.full(len(KP_NAMES), np.nan)
    return np.linalg.norm(pred_px - gt_px, axis=1) / inter_ocular

In [18]:
# Test split processed by 5-1_Keypoint_Regression_Model: normalized ground truth,
# original image size, and the source image in S3
kp_test = pd.read_parquet(io.BytesIO(read_s3_bytes(f"s3://{bucket}/{kp_test_data_prefix}/annotations.parquet")))

rows = []
for _, row in tqdm(kp_test.iterrows(), total=len(kp_test), desc="Keypoint baseline traffic", unit="image"):
    try:
        inference_id, result = invoke(kp_predictor, row["remote_path"])
    except s3.exceptions.NoSuchKey:
        print(row["remote_path"])
        continue

    pred = [result[name][axis] for name in KP_NAMES for axis in ("x", "y")]
    nme = keypoint_nme(pred, row[KP_COLS_NORM].to_numpy(), row["width"], row["height"])
    rows.append({
        "inference_id": inference_id,
        "remote_path": row["remote_path"],
        **dict(zip(KP_COLS_PRED, pred)),
        # Predicted eye distance in normalized coordinates: a label-free proxy for face size
        "pred_inter_ocular": float(np.hypot(pred[0] - pred[2], pred[1] - pred[3])),
        "nme": float(np.mean(nme)),
        **{f"nme_{name}": float(value) for name, value in zip(KP_NAMES, nme)},
        **result["input_stats"],
    })

kp_baseline_df = pd.DataFrame(rows)
kp_baseline_df["nme"].describe()

Keypoint baseline traffic:   0%|          | 0/993 [00:00<?, ?image/s]

Keypoint baseline traffic: 100%|██████████| 993/993 [01:59<00:00,  8.29image/s]


count    993.000000
mean       0.304778
std        1.888692
min        0.038503
25%        0.124015
50%        0.175034
75%        0.252350
max       55.946028
Name: nme, dtype: float64

In [19]:
# Train/serve consistency: pipeline evaluation vs. the deployed endpoint on the same test split.
# The pipeline scores preprocessed images; the endpoint resizes the original JPEG itself.
offline = registered_evaluation(kp_model_package_arn)
valid_nme = kp_baseline_df["nme"].dropna()

kp_endpoint_metrics = {
    "nme_mean": valid_nme.mean(),
    "nme_median": valid_nme.median(),
    "nme_p95": valid_nme.quantile(0.95),
    "failure_rate_at_0_1": (valid_nme > 0.1).mean(),
}

display(pd.DataFrame({
    "pipeline_evaluation": {name: offline["regression_metrics"][name]["value"] for name in kp_endpoint_metrics},
    "endpoint": kp_endpoint_metrics,
}).round(4))

pd.DataFrame({
    "pipeline_evaluation": offline["per_keypoint_nme"],
    "endpoint": {name: kp_baseline_df[f"nme_{name}"].mean() for name in KP_NAMES},
}).round(4)

,pipeline_evaluation,endpoint
nme_mean,0.3003,0.3048
nme_median,0.1703,0.1750
nme_p95,0.5311,0.5279
failure_rate_at_0_1,0.8731,0.8832


,pipeline_evaluation,endpoint
left_eye,0.2713,0.2797
right_eye,0.2532,0.2581
mouth,0.2997,0.3034
left_ear_1,0.3139,0.3156
left_ear_2,0.3403,0.3407
left_ear_3,0.2941,0.2949
right_ear_1,0.2873,0.2939
right_ear_2,0.3312,0.3385
right_ear_3,0.3115,0.3183


### Model-quality baseline
Model Monitor's model-quality jobs compare one predicted value with one label, so they cannot score 9 keypoints at once. The keypoint quality baseline uses NME instead. It is written in Model Monitor's statistics/constraints layout, so a custom monitoring job can check live NME against it once ground truth arrives. As with Model Monitor's suggested constraints, each baseline value is its own threshold.

In [20]:
kp_baseline_prefix = f"{baseline_prefix}/keypoint"

kp_quality_statistics = {
    "version": 0.0,
    "dataset": {
        "item_count": int(valid_nme.size),
        "excluded_zero_inter_ocular": int(kp_baseline_df["nme"].isna().sum()),
    },
    "keypoint_metrics": {name: {"value": float(value)} for name, value in kp_endpoint_metrics.items()},
    "per_keypoint_nme": {name: {"value": float(kp_baseline_df[f"nme_{name}"].mean())} for name in KP_NAMES},
}

kp_quality_constraints = {
    "version": 0.0,
    "keypoint_constraints": {
        name: {"threshold": float(value), "comparison_operator": "GreaterThanThreshold"}
        for name, value in kp_endpoint_metrics.items()
    },
}

for name, body in [("statistics.json", kp_quality_statistics), ("constraints.json", kp_quality_constraints)]:
    s3.put_object(
        Bucket=bucket,
        Key=f"{kp_baseline_prefix}/model-quality/{name}",
        Body=json.dumps(body, indent=2).encode(),
    )

pd.DataFrame(kp_quality_constraints["keypoint_constraints"]).T

,threshold,comparison_operator
nme_mean,0.304778,GreaterThanThreshold
nme_median,0.175034,GreaterThanThreshold
nme_p95,0.527936,GreaterThanThreshold
failure_rate_at_0_1,0.883182,GreaterThanThreshold


In [21]:
# Input/prediction drift baseline dataset: request-image statistics, predicted coordinates and
# eye distance, no labels needed. Alphabetical, matching src/monitoring/keypoint_record_preprocessor.py
KP_DRIFT_FEATURES = sorted(KP_COLS_PRED + ["pred_inter_ocular"] + INPUT_STAT_NAMES)
kp_prediction_dataset_uri = upload_csv(
    kp_baseline_df[KP_DRIFT_FEATURES].astype({"width": "int64", "height": "int64"}),
    f"{kp_baseline_prefix}/prediction-dataset.csv",
)

# Inference log: which inference ID was sent with which image, plus its per-image error
upload_csv(kp_baseline_df, f"{kp_baseline_prefix}/inference-log.csv")

's3://sagemaker-us-east-1-298748835671/cat-landmarks-project/monitoring/19d31439-87ca-4ff9-87e8-3327a4316c7f/baselines/keypoint/inference-log.csv'

### Input and prediction drift baseline
Distributions of the request images and of the predicted keypoint coordinates and eye distance. A shift flags changes in image quality, framing or face size without waiting for labels.

In [22]:
kp_prediction_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

kp_prediction_monitor.suggest_baseline(
    job_name=f"kp-prediction-baseline-{endpoint_id}",
    baseline_dataset=kp_prediction_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{kp_baseline_prefix}/prediction-drift",
    wait=True,
    logs=False,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name kp-prediction-baseline-19d31439-87ca-4ff9-87e8-3327a4316c7f


.........................................................................................!

In [23]:
kp_prediction_baseline_job = kp_prediction_monitor.latest_baselining_job

summarize_feature_statistics(kp_prediction_baseline_job)

,type,mean,std_dev,min,max
feature,,,,,
aspect_ratio,Fractional,1.222708,0.287569,0.408000,2.306306
brightness,Fractional,0.436151,0.119130,0.043037,0.908183
contrast,Fractional,0.235893,0.055180,0.069294,0.408068
height,Integral,521.087613,206.795479,192.000000,1024.000000
left_ear_1_x_pred,Fractional,0.353742,0.148789,0.038008,0.802100
left_ear_1_y_pred,Fractional,0.343593,0.119372,0.073331,0.795439
left_ear_2_x_pred,Fractional,0.338177,0.178098,-0.089967,0.964232
left_ear_2_y_pred,Fractional,0.178614,0.153025,-0.245235,0.840850
left_ear_3_x_pred,Fractional,0.439448,0.140432,0.086109,0.896003


In [24]:
# Baseline locations for the monitoring schedules
%store monitoring_prefix data_capture_prefix baseline_prefix

Stored 'monitoring_prefix' (str)
Stored 'data_capture_prefix' (str)
Stored 'baseline_prefix' (str)


# Monitoring Schedules
Production traffic, ground truth and the scheduled monitors that use these baselines are set up in `6-1_Monitoring_Schedules.ipynb`.

# Deleting endpoints
Run the imports cell at the top, then set `DELETE_ENDPOINTS = True` and run the cell below to delete both endpoints and their configurations. Endpoint names come from `%store`, so it works from a fresh kernel. Delete the monitoring schedules first (end of `6-1_Monitoring_Schedules`), because an endpoint can't be deleted while a schedule still watches it.

In [25]:
%store -r

DELETE_ENDPOINTS = False

if DELETE_ENDPOINTS:
    sm = boto3.client("sagemaker")
    for name in [kp_endpoint_name, cls_endpoint_name]:
        sm.delete_endpoint(EndpointName=name)
        # Model.deploy names the endpoint configuration after the endpoint
        sm.delete_endpoint_config(EndpointConfigName=name)

    print("Remaining endpoints:", [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]])